In [1]:
using LowLevelFEM, LinearAlgebra

In [2]:
openGeometry("rectangles.geo")

In [3]:
#openPreProcessor()

In [4]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u);

In [5]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0)
bc_top = BoundaryCondition("top", ux=0, uy=(x,y,z)->x*(x-10)/130)

K = ∫(SymGrad(U) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

#showDoFResults(u, name="u", factor=1, visible=true)

nodal VectorField
[0.0; 0.0; … ; -0.026961545388719164; -0.17253959505900868;;]


## Penalty contact

The contact object contains only the contact geometry and kinematics. The full
contact operator maps the displacement field to the local contact space,

\[
G: V_u \rightarrow V_c ,
\]

while \(P_a\) selects the currently active contact nodes,

\[
G_a=P_aG.
\]

The penalty surface operator is assembled once with the ordinary LLFEM weak-form
machinery and then restricted to the slave surface. In 2D the contact-space
ordering is \((n,t)\). For frictionless contact \(c_t=0\).


In [6]:

r = nodePositionVector(U)

C = contact(
    u,
    master="master",
    slave="slave"
)

cn = 1e6
ct = 0.0

Dc = [cn 0.0
      0.0 ct]

C0 = ∫(U ⋅ Dc ⋅ U, Γ="slave")
Cc = subSystemMatrix(C0; onPhysicalGroup="slave");



At a fixed contact geometry the active penalty contribution is

\[
d_a=P_a d,\qquad
C_a=P_a C_c P_a^T,\qquad
G_a=P_aG,
\]

\[
r_c=G_a^T C_a d_a,\qquad
K_c=G_a^T C_aG_a.
\]

Since \(d=G(r+u)\), freezing the current geometry gives the Newton equation

\[
(K+K_c)u_{\mathrm{new}}=f-K_cr.
\]

Thus the linear correction can still be solved with the ordinary `solveField`
function. A line search is retained because the projection and active set may
change during the iteration.


In [7]:

support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

# Only used to compare nonlinear residuals on unconstrained DoFs.
freeNorm(v::VectorField) = LinearAlgebra.norm(elementsToNodes(v).a[free, 1])

u_it = copy(u)

old_tags = copy(C.master_element_tags)
old_G = copy(C.G)

for iter in 1:30

    updateContact!(C, u_it)

    nchanged = count(old_tags .!= C.master_element_tags)

    dG = norm(C.G - old_G) /
         max(norm(old_G), eps())

    old_tags = copy(C.master_element_tags)
    old_G = copy(C.G)

    # Active contact algebra
    Ga = C.Pa * C.G
    Ca = C.Pa * Cc * C.Pa'
    da = C.Pa * C.d

    # Contact residual and frozen-geometry tangent
    rc = Ga' * (Ca * da)
    Kc = Ga' * Ca * Ga

    # Total residual
    R = K * u_it - f + rc
    R0 = freeNorm(R)

    # Full frozen-geometry Newton step:
    # (K + Kc) u_new = f - Kc r
    Δu = solveField(
        K + Kc,
        -R,
        support=support
    )

    u_new = u_it + Δu
    Δu = u_new - u_it

    # Line search because G, projection and the active set change with u.
    α = 1.0
    u_trial = copy(u_it)
    Rtrial = R

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(C, u_trial)

        Ga_trial = C.Pa * C.G
        Ca_trial = C.Pa * Cc * C.Pa'
        da_trial = C.Pa * C.d

        rc_trial = Ga_trial' * (Ca_trial * da_trial)

        Rtrial = K * u_trial - f + rc_trial

        freeNorm(Rtrial) < R0 && break

        α *= 0.5
    end

    u_it = u_trial

    err = freeNorm(α * Δu) /
          max(freeNorm(u_it), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(C.active),
        ", master changes = ", nchanged,
        ", dG = ", dG,
        ", min gap = ", minimum(C.gap_values),
        ", error = ", err,
        ", |R| = ", freeNorm(Rtrial)
    )

    err < 1e-8 && break
end

u = u_it

# Synchronize the stored contact state with the converged field.
updateContact!(C, u);


iter = 1, α = 1.0, active = 83, master changes = 0, dG = 0.0, min gap = -0.031101590280243344, error = 0.47606683968365315, |R| = 13502.282320492564
iter = 2, α = 1.0, active = 85, master changes = 72, dG = 0.45051598880040883, min gap = -0.024133802912512808, error = 0.31773590672484836, |R| = 1792.2693692613068
iter = 3, α = 1.0, active = 87, master changes = 42, dG = 0.31775872480219514, min gap = -0.03417665647403004, error = 0.24137192844801372, |R| = 1428.9253201560527
iter = 4, α = 1.0, active = 87, master changes = 20, dG = 0.3875413988148177, min gap = -0.04435483873857029, error = 0.1932354715291275, |R| = 1037.6895450530144
iter = 5, α = 0.5, active = 87, master changes = 36, dG = 0.2370666004734868, min gap = -0.049457114594686, error = 0.0877775327458507, |R| = 1005.6496541198792
iter = 6, α = 0.25, active = 87, master changes = 6, dG = 0.25294818411357656, min gap = -0.05200553048466248, error = 0.04197872658043687, |R| = 920.9858632408975
iter = 7, α = 9.5367431640625e-7

In [8]:

showDoFResults(u, name="u", factor=1, visible=true)


0


## Contact fields

`C.d` is a reduced `ContactVector`. Mapping it back to the displacement mesh
makes the local contact components available through the ordinary field API.

For the current closest-point geometry, `D[1]` is the normal gap. The tangential
component of the current position difference is approximately zero by
construction; tangential slip will later be accumulated from displacement
increments/history.


In [9]:

D = VectorField(C.d)

gap = D[1]

# Active normal gap, expanded back to the full contact space.
da_full = C.Pa' * (C.Pa * C.d)
Da = VectorField(da_full)

# Pointwise penalty traction (positive in compression).
pressure = -cn * Da[1]

gap

plotOnBeam("slave", nodesToElements(pressure))
plotOnBeam("slave", nodesToElements(gap))


2

In [10]:

# Example postprocessing:
# showElementResults(nodesToElements(gap), name="gap", visible=true)
# showElementResults(nodesToElements(pressure), name="pressure", visible=true)

openPostProcessor()


XOpenIM() failed
Fontconfig warning: using without calling FcInit()
